# שלב 06 — השוואה אזורית

האם הפגיעוּת שונה בין אזורי הארץ? מחלקים את התחנות לפי אזור (צפון/מרכז/דרום/ירושלים) ומשווים: כמה תחנות קריטיות, כמה Articulation Points, ומה ה‑Degree הממוצע בכל אזור.

**תחנה קריטית** מוגדרת כאן כתחנה שהיא Articulation Point או שה‑Betweenness שלה בעשירון העליון.

In [ ]:
# התקנת הספריות הנדרשות (להריץ פעם אחת; אפשר לדלג אם כבר מותקנות)
%pip install pandas numpy matplotlib seaborn python-bidi

In [ ]:
from pathlib import Path
import pickle, json
import pandas as pd
import numpy as np
import networkx as nx


def find_repo_root(start: Path) -> Path:
    for cand in [start.resolve(), *start.resolve().parents]:
        if (cand / "israel-public-transportation").exists():
            return cand
    raise FileNotFoundError("repo root not found - set ROOT manually")


ROOT = find_repo_root(Path.cwd())
BASE = ROOT / "public_transport_network_notebooks"
METRICS_CSV = BASE / "outputs" / "04_centrality_analysis" / "stop_metrics.csv"
AP_CSV = BASE / "outputs" / "03_network_descriptive_analysis" / "articulation_points.csv"
BRIDGES_CSV = BASE / "outputs" / "03_network_descriptive_analysis" / "bridges.csv"
OUT_DIR = BASE / "outputs" / "06_regional_comparison"
FIG_DIR = BASE / "figures" / "06_regional_comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR:", OUT_DIR)

In [ ]:
import re
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from bidi.algorithm import get_display


def _fix(t):
    """מסדר טקסט עברי לתצוגה נכונה (bidi). אנגלית ומספרים נשארים כמו שהם."""
    if isinstance(t, str) and any(0x590 <= ord(c) <= 0x5FF for c in t):
        return get_display(t)
    return t


import matplotlib.text as _mt
if not getattr(_mt.Text, "_bidi", False):
    _orig = _mt.Text.set_text
    def _set(self, s):
        if isinstance(s, str) and getattr(self, "_disp", None) == s:
            return _orig(self, s)
        f = _fix(s)
        if isinstance(f, str):
            self._disp = f
        return _orig(self, f)
    _mt.Text.set_text = _set
    _mt.Text._bidi = True

sns.set_theme(style="whitegrid", font_scale=1.1)
matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
print("עברית בגרפים מופעלת")

## חישוב סיכום אזורי

In [ ]:
metrics = pd.read_csv(METRICS_CSV, encoding="utf-8-sig")
ap_df = pd.read_csv(AP_CSV, encoding="utf-8-sig")

ap_set = set(ap_df["stop_id"].astype(str))
metrics["is_ap"] = metrics["stop_id"].astype(str).isin(ap_set)
metrics["betweenness"] = pd.to_numeric(metrics["betweenness"], errors="coerce").fillna(0)
metrics["degree"] = pd.to_numeric(metrics["degree"], errors="coerce").fillna(0)

btw_p90 = metrics["betweenness"].quantile(0.90)
metrics["is_critical"] = metrics["is_ap"] | (metrics["betweenness"] >= btw_p90)

region_summary = metrics.groupby("region").agg(
    total_stops=("stop_id", "count"),
    critical_stops=("is_critical", "sum"),
    ap_stops=("is_ap", "sum"),
    avg_degree=("degree", "mean"),
    avg_betweenness=("betweenness", "mean"),
    max_betweenness=("betweenness", "max"),
).reset_index()
region_summary["pct_critical"] = (region_summary["critical_stops"] / region_summary["total_stops"] * 100).round(1)
region_summary["pct_ap"] = (region_summary["ap_stops"] / region_summary["total_stops"] * 100).round(2)

region_summary.to_csv(OUT_DIR / "regional_summary.csv", index=False, encoding="utf-8-sig")
metrics.to_csv(OUT_DIR / "stops_with_region.csv", index=False, encoding="utf-8-sig")
region_summary

## גרפים אזוריים

In [ ]:
colors = ["#2563eb", "#dc2626", "#16a34a", "#d97706"]
regions = region_summary.sort_values("pct_critical", ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col, title in zip(axes, ["pct_critical", "pct_ap", "avg_degree"],
                          ["% תחנות קריטיות", "% Articulation Points", "Degree ממוצע"]):
    ax.bar(region_summary["region"], region_summary[col], color=colors)
    ax.set_title(title)
plt.suptitle("השוואה אזורית — רשת התחבורה הציבורית")
plt.tight_layout()
plt.savefig(FIG_DIR / "regional_vulnerability_comparison.png", dpi=150)
plt.show()

In [ ]:
df_map = metrics.dropna(subset=["lat", "lon"])
region_colors = {"מרכז": "#2563eb", "צפון": "#16a34a", "דרום": "#dc2626", "ירושלים": "#d97706"}
fig, ax = plt.subplots(figsize=(8, 11))
for reg, grp in df_map.groupby("region"):
    ax.scatter(grp["lon"], grp["lat"], s=2, alpha=0.3, color=region_colors.get(reg, "#6b7280"), label=reg)
critical = df_map[df_map["is_critical"]]
ax.scatter(critical["lon"], critical["lat"], s=15, color="black", alpha=0.7, zorder=5, label="קריטי")
ax.legend(markerscale=3, fontsize=9)
ax.set_title("מפת תחנות לפי אזור + תחנות קריטיות (שחור)")
ax.set_xlabel("קו אורך"); ax.set_ylabel("קו רוחב")
plt.tight_layout()
plt.savefig(FIG_DIR / "stations_map_by_region.png", dpi=150)
plt.show()